In [ ]:
# LIQUIDITY WATCH | NB_02_Macro_Context  (lakehouse par défaut : LH_LiquidityWatch)
# Charge le contexte de marché quotidien, gratuit et sans clé : le taux €STR de la BCE et l'indice VIX du CBOE.
import io
import requests
import pandas as pd

H = {"User-Agent": "LiquidityWatch/1.0 (Microsoft Fabric)"}
ESTR_URL = "https://data-api.ecb.europa.eu/service/data/EST/B.EU000A2X2A25.WT?format=csvdata&startPeriod=2024-01-01"
VIX_URL = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv"


def get_csv(url):
    r = requests.get(url, headers=H, timeout=30)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))


# €STR : le taux au jour le jour du marché monétaire en euros, publié chaque jour ouvré par la BCE
estr = get_csv(ESTR_URL)[["TIME_PERIOD", "OBS_VALUE"]]
estr.columns = ["rate_date", "estr_pct"]
estr["rate_date"] = pd.to_datetime(estr["rate_date"])
estr = estr.sort_values("rate_date")
estr["estr_change_bp"] = (estr["estr_pct"].diff() * 100).round(1)

# VIX : la volatilité attendue sur le S&P 500, le thermomètre de la peur des marchés
vix = get_csv(VIX_URL)
vix.columns = [c.strip().lower() for c in vix.columns]
vix["date"] = pd.to_datetime(vix["date"], format="%m/%d/%Y")
vix = vix[vix["date"] >= "2024-01-01"].sort_values("date")
vix = vix.rename(columns={"date": "vix_date", "close": "vix_close"})[["vix_date", "open", "high", "low", "vix_close"]]
vix["regime"] = pd.cut(vix["vix_close"], [0, 15, 20, 30, 1000], labels=["Calme", "Normal", "Tendu", "Stress"]).astype(str)

for name, df in [("macro_estr", estr), ("macro_vix", vix)]:
    spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    last = df.iloc[-1]
    print(f"{name} : {len(df)} lignes, dernière date {last.iloc[0]:%d/%m/%Y}")
print(f"VIX {vix['vix_close'].iloc[-1]:.2f} ({vix['regime'].iloc[-1]}) · €STR {estr['estr_pct'].iloc[-1]:.3f} %")